In [119]:
# ================================
# 1. INSTALL & IMPORT LIBRARIES
# ================================
# pip install pandas numpy scikit-learn nltk contractions emoji

import pandas as pd
import numpy as np
import re
import contractions
import emoji
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PARAS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\PARAS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [120]:
# ================================
# 2. LOAD DATA
# ================================
train_df = pd.read_csv("data/twitter_training.csv", header=None)
train_df.columns = ['id', 'entity', 'label', 'text']

# Keep only needed columns
train_df = train_df[['text', 'label']]

# Remove nulls & duplicates
train_df.dropna(inplace=True)
train_df.drop_duplicates(inplace=True)

# Remove "Irrelevant"
train_df = train_df[train_df['label'] != 'Irrelevant']

In [121]:
# ================================
# 3. ENCODE LABELS (IMPORTANT)
# ================================
mapping = {'Negative': 0, 'Positive': 1, 'Neutral': 2}
train_df['label'] = train_df['label'].map(mapping)

In [122]:
# ================================
# 4. TEXT CLEANING FUNCTION
# ================================
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = contractions.fix(text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = emoji.replace_emoji(text, replace="")
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]

    return " ".join(words)

train_df['clean_text'] = train_df['text'].apply(clean_text)

In [131]:
# ================================
# 5. TRAIN-VALIDATION SPLIT
# ================================
X_train, X_val, y_train, y_val = train_test_split(
    train_df['clean_text'],
    train_df['label'],
    test_size=0.2,
    stratify=train_df['label'],
    random_state=42
)

In [132]:
# ================================
# 6. TF-IDF VECTORIZATION
# ================================
tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9
)

X_train = tfidf.fit_transform(X_train)
X_val = tfidf.transform(X_val)

In [133]:
# ================================
# 7. MODEL TRAINING (LOGISTIC REGRESSION)
# ================================
param_grid = {
    'C': [0.01, 0.1, 1, 10]
}

lr = LogisticRegression(max_iter=300, class_weight='balanced')

grid = GridSearchCV(
    lr,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

model = grid.best_estimator_

print("Best Params:", grid.best_params_)
print("CV Score:", grid.best_score_)

Best Params: {'C': 10}
CV Score: 0.7144471740226764


In [134]:
# ================================
# 8. EVALUATION
# ================================
y_pred = model.predict(X_val)

print("\nValidation Accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_val, y_pred))


Validation Accuracy: 0.7221883969731234

Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.75      0.76      4247
           1       0.73      0.73      0.73      3828
           2       0.65      0.68      0.67      3422

    accuracy                           0.72     11497
   macro avg       0.72      0.72      0.72     11497
weighted avg       0.72      0.72      0.72     11497


Confusion Matrix:
 [[3195  447  605]
 [ 419 2788  621]
 [ 516  586 2320]]


In [135]:
# ================================
# 9. SAVE MODEL
# ================================
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(tfidf, open("tfidf.pkl", "wb"))

In [136]:
# ================================
# 10. APPLY ON INSTAGRAM DATA
# ================================
insta_df = pd.read_csv("data/Instagram data.csv", encoding='latin1')

# Feature Engineering
insta_df['engagement'] = (
    insta_df['Likes'] +
    insta_df['Comments'] +
    insta_df['Shares'] +
    insta_df['Saves']
)

insta_df['engagement_rate'] = insta_df['engagement'] / insta_df['Impressions']

# Clean captions
insta_df['clean_text'] = insta_df['Caption'].astype(str).apply(clean_text)

# Transform using same TF-IDF
X_insta = tfidf.transform(insta_df['clean_text'])

# Predict sentiment
insta_df['sentiment'] = model.predict(X_insta)

# Decode labels
reverse_mapping = {0: "Negative", 1: "Positive", 2: "Neutral"}
insta_df['sentiment_label'] = insta_df['sentiment'].map(reverse_mapping)

In [137]:
# ================================
# 11. ANALYSIS
# ================================

# Top posts
top_posts = insta_df.sort_values(by='engagement', ascending=False)
print("\nTop Posts:\n", top_posts[['Caption', 'engagement']].head())

# Source analysis
source_cols = ['From Home', 'From Hashtags', 'From Explore', 'From Other']
print("\nTraffic Sources:\n", insta_df[source_cols].sum())

# Correlation
print("\nCorrelation:\n", insta_df[['Likes', 'Comments', 'Shares', 'Saves', 'Impressions']].corr())

# Caption length analysis
insta_df['caption_length'] = insta_df['Caption'].apply(lambda x: len(str(x).split()))
print("\nCaption vs Engagement:\n", insta_df[['caption_length', 'engagement']].corr())



Top Posts:
                                                Caption  engagement
117  Here are some of the best data science certifi...        1721
118  175 Python Projects with Source Code solved an...        1127
49   Here are some of the best websites that you ca...        1045
90   Here are some of the best websites that you ca...        1045
114  Here are some of the best data science certifi...         986

Traffic Sources:
 From Home        294619
From Hashtags    224614
From Explore     128294
From Other        20360
dtype: int64

Correlation:
                 Likes  Comments    Shares     Saves  Impressions
Likes        1.000000  0.123586  0.707794  0.845643     0.849835
Comments     0.123586  1.000000  0.016933 -0.026912    -0.028524
Shares       0.707794  0.016933  1.000000  0.860324     0.634675
Saves        0.845643 -0.026912  0.860324  1.000000     0.779231
Impressions  0.849835 -0.028524  0.634675  0.779231     1.000000

Caption vs Engagement:
                 caption_len

In [138]:
# ================================
# 12. SAVE OUTPUT
# ================================
insta_df.to_csv("output/final_output.csv", index=False)

print("\n✅ Pipeline Completed Successfully")


✅ Pipeline Completed Successfully
